# Auditar a qualidade dos dados

Você recebeu uma tabela de viagens. Em uma linha, a distância é zero. Em outra, está vazia. O que cada caso significa para quem analisa?

Use pandas e retome [Chaves e junções](https://devrgs.github.io/Aprendendo-linguagens/#/python/pandas/juntar-tabelas). As tabelas são declaradas no código.

Esquema é a descrição de campos, tipos e unidades. Aqui `id` é texto, `km` é distância e deve ser não negativa quando presente. `isna()` detecta ausência. Zero é um número dentro do domínio. `duplicated()` marca ocorrências excedentes de uma linha idêntica. Se a mesma chave identifica duas linhas diferentes, há um conflito que precisa de apuração.

Execute as células na ordem com **Shift+Enter**. A preparação cria uma pasta nova na área de arquivos do caderno. Baixe o caderno editado e os resultados que quiser guardar.


In [ ]:
from pathlib import Path as _Path
from tempfile import mkdtemp as _mkdtemp
import os as _os
import sys as _sys
from importlib import invalidate_caches as _refresh_imports

# Auxiliares editáveis devem ser relidos sem aproveitar bytecode antigo.
_sys.dont_write_bytecode = True

_raiz = _Path.cwd()
_trabalho = _Path(_mkdtemp(prefix='python-qualidade-dos-dados-', dir=_raiz))
import numpy
import pandas
import matplotlib


## E1 · Preveja e execute

Examine as linhas a:0, b:ausente, a:0. Liste IDs com ausência, com zero e de ocorrências repetidas.


In [ ]:
_pasta = _trabalho / 'e1'
_pasta.mkdir(exist_ok=True)
_os.chdir(_pasta)
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]
if str(_pasta) not in _sys.path:
    _sys.path.insert(0, str(_pasta))
_refresh_imports()


In [ ]:
import pandas as pd

df = pd.DataFrame({"id": ["a", "b", "a"], "km": [0, None, 0]})
print("Ausentes:", df.loc[df["km"].isna(), "id"].tolist())
print("Zeros:", df.loc[df["km"].eq(0), "id"].tolist())
print("Duplicatas excedentes:", df.loc[df.duplicated(), "id"].tolist())


In [ ]:
# Voltar à pasta do caderno para a etapa seguinte.
_os.chdir(_raiz)
if str(_pasta) in _sys.path:
    _sys.path.remove(str(_pasta))
# Liberar módulos locais para a próxima etapa importar sua própria versão.
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]


Saída esperada

```text
Ausentes: ['b']
Zeros: ['a', 'a']
Duplicatas excedentes: ['a']
```

A ausência é uma propriedade da célula. A duplicata descreve a linha excedente.


## E2 · Altere ou depure

Inclua também a:3. Remova cópias exatas apenas numa tabela temporária. Aponte os IDs conflitantes.


In [ ]:
_pasta = _trabalho / 'e2'
_pasta.mkdir(exist_ok=True)
_os.chdir(_pasta)
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]
if str(_pasta) not in _sys.path:
    _sys.path.insert(0, str(_pasta))
_refresh_imports()


In [ ]:
import pandas as pd

df = pd.DataFrame({"id": ["a", "b", "a", "a"], "km": [0, None, 0, 3]})
unicas = df.drop_duplicates()
conflitos = unicas.loc[unicas["id"].duplicated(keep=False), "id"].unique()
print(conflitos.tolist())


In [ ]:
# Voltar à pasta do caderno para a etapa seguinte.
_os.chdir(_raiz)
if str(_pasta) in _sys.path:
    _sys.path.remove(str(_pasta))
# Liberar módulos locais para a próxima etapa importar sua própria versão.
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]


Saída esperada

```text
['a']
```

Depois de retirar a repetição exata, a ainda aponta para duas linhas com distâncias diferentes.


## E3 · Resolva

Audite ausências, negativos, duplicatas excedentes e conflitos de ID sem alterar o original. Use a:0, b:ausente, c:-2, a:0 e a:1. No caso de borda c:-2, mantenha a linha disponível para inspeção e só liste no relatório.


In [ ]:
_pasta = _trabalho / 'e3'
_pasta.mkdir(exist_ok=True)
_os.chdir(_pasta)
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]
if str(_pasta) not in _sys.path:
    _sys.path.insert(0, str(_pasta))
_refresh_imports()


In [ ]:
import pandas as pd

def auditar(df):
    unicas = df.drop_duplicates()
    conflitos = unicas.loc[unicas["id"].duplicated(keep=False), "id"].unique()
    return {
        "km ausentes": df.loc[df["km"].isna(), "id"].tolist(),
        "km negativos": df.loc[df["km"].lt(0), "id"].tolist(),
        "duplicatas excedentes": df.loc[df.duplicated(), "id"].tolist(),
        "IDs em conflito": sorted(conflitos.tolist()),
    }

df = pd.DataFrame({"id": ["a", "b", "c", "a", "a"], "km": [0, None, -2, 0, 1]})
for regra, ids in auditar(df).items():
    print(regra, len(ids), ids)
assert len(df) == 5
assert df.loc[2, "km"] == -2


In [ ]:
# Voltar à pasta do caderno para a etapa seguinte.
_os.chdir(_raiz)
if str(_pasta) in _sys.path:
    _sys.path.remove(str(_pasta))
# Liberar módulos locais para a próxima etapa importar sua própria versão.
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]


Saída esperada

```text
km ausentes 1 ['b']
km negativos 1 ['c']
duplicatas excedentes 1 ['a']
IDs em conflito 1 ['a']
```

As três primeiras medidas contam linhas brutas. A quarta conta chaves distintas em conflito. A própria tabela permanece disponível para decidir o destino de cada linha. Com a qualidade descrita, avance para distribuições.
